# **ETL BRONZE PROCESS**

In [0]:
%pip install pypdf

In [0]:
import os
import base64
from databricks.sdk import WorkspaceClient
from openai import OpenAI
from pypdf import PdfReader
import json
from pyspark.sql.functions import lit, current_timestamp

wkc = WorkspaceClient()

prompt= """

1. FORMATO DE SALIDA (JSON)
La respuesta debe ser únicamente JSON válido, con la siguiente estructura exacta:

	json
	[
	 {
		"nombre_mercado": "null",
		"nombre_producto": "null",
		"unidad_medida": "null",
		"inicio_fecha_precios": "null",
		"fin_fecha_precios": "null",
		"precio_inicio": "null",
		"precio_fin": "null",
		"error": "null"
	 }
	]
Regla crítica: Todos los valores deben ir como strings (entre comillas dobles), incluyendo "null" cuando no haya dato.

2. REGLAS DE EXTRACCIÓN DE CAMPOS
2.1 nombre_mercado
	Tomar el texto que está dentro del rectángulo amarillo (título del mercado/feria/supermercado).

	Normalizar contra la lista oficial de mercados comunes (minúsculas, con tildes correctas).

	Si el texto no coincide con la lista, mantener el texto tal cual se leyó.

	Si no se puede leer, colocar "null" y reportar en "error".

	Lista oficial de mercados normalizados:

	- mercado zonal belén, comayagüela
	- ahorro ferias del pueblo
	- feria del agricultor y el artesano
	- supermercados tegucigalpa
	- supermercados san pedro sula
	- mercado el rápido san pedro sula
	- feria agropecuaria de villanueva
	- mercado san isidro, las americas y colón

2.2 nombre_producto
	Usar el catálogo estándar de productos según la posición de la fila en la tabla.
	Si la posición no coincide con el catálogo, intentar leer el texto directamente.
	Si no se puede leer, colocar "null" y reportar en "error".

2.3 unidad_medida
	Usar el catálogo estándar de unidades según la posición de la fila.
	Si no se puede determinar, colocar "null" y reportar en "error".

2.4 inicio_fecha_precios y fin_fecha_precios
	Buscar el texto "( Semana del ...)"* o "Semana del ..." en la parte superior de las columnas.
	Dividirlo en dos fechas con formato DD-MM-AAAA:
	inicio_fecha_precios: primer valor de la semana
	fin_fecha_precios: segundo valor de la semana

2.5 precio_inicio
	Tomar el valor de la primera columna de "Precios L" (o "Precio L. [fecha inicio]").

2.6 precio_fin
	Tomar el valor de la segunda columna de "Precios L" (o "Precio L. [fecha fin]").

2.7 error
	Si no hay problemas: "null"
	Si hay problemas de lectura: indicar número de página + nombre del documento.
	Formato exacto: "pag X - nombre_del_archivo.pdf"

	3. REGLAS DE PROCESAMIENTO DE PÁGINAS Y TABLAS
	Procesar todas las páginas del archivo, sin excepción, desde la página 1 hasta la última del archivo.
	Incluir todas las tablas que aparezcan en cada página, aunque estén parcialmente cortadas, borrosas o incompletas.
	Si una tabla no se puede leer en absoluto:
	Crear una sola entrada JSON para esa tabla.
	Poner "null" en todos los campos.
	Indicar en "error" el número de página + nombre del documento.
	Si una tabla se puede leer parcialmente:
	Incluir una entrada JSON por cada fila que se pueda leer.
	En las filas donde falte algún valor, poner "null" en ese campo.
	En el campo "error" de esa fila, indicar el número de página + nombre del documento.
	Nunca omitir una tabla por estar borrosa, cortada o con texto corrupto.
	Si una página contiene solo ruido o texto sin estructura de tabla:
	No se genera entrada de tabla.
	Se debe reportar con una entrada JSON con todos los campos "null" y en "error" el número de página + nombre del documento.
	No agrupar ni resumir: cada fila de cada tabla debe ser un objeto JSON independiente.
	Si no se puede leer un valor de una fila de la tabla, colocar "null" en ese campo (no omitirlo) y reportar en "error" el número de página + nombre del documento.

4. CATÁLOGO ESTÁNDAR DE PRODUCTOS Y UNIDADES
	Usar este catálogo según la posición de la fila en la tabla (1 a 30):

	#	Producto	Unidad
	1	Tajo de Res	libra
	2	Costilla de Res Regular	libra
	3	Costilla de Cerdo Regular	libra
	4	Pollo Entero Congelado sin Menudos	libra
	5	Pescado Blanco	libra
	6	Leche Integra en Polvo	360 g
	7	Leche pasteurizada fluida en bolsa	0.946l
	8	Mantequilla	libra
	9	Queso Blanco Fresco	libra
	10	Huevo Mediano	cartón
	11	Cebolla Amarilla	libra
	12	Tomate Pera	libra
	13	Papas	libra
	14	Yuca	libra
	15	Repollo	libra
	16	Plátano Maduro	unidad
	17	Naranja Dulce	unidad
	18	Banano Fresco Maduro	unidad
	19	Frijol Rojo a Granel	libra
	20	Arroz Clasificado a Granel	libra
	21	Espagueti	200 g
	22	Azúcar Blanca	libra
	23	Café Molido	16 oz
	24	Salsa de Tomate	400 g
	25	Aceite Vegetal	443 ml
	26	Manteca Comestible de Origen Vegetal	libra
	27	Sal común de Mesa Yodada	227 g
	28	Tortilla de Maíz	unidad
	29	Pan Molde Blanco	540 g
	30	Jugo de Naranja	500ml
	Regla adicional: Cuando una tabla no tenga exactamente 30 filas, usar la posición de cada fila para asignar el nombre y unidad correctos según este catálogo, en lugar de depender del texto ilegible.

5. CORRECCIONES COMUNES DE LECTURA (APLICAR AUTOMÁTICAMENTE)
5.1 Productos
	Texto ilegible	Corrección
	Cotilla	Costilla
	Mantegulla	Mantequilla
	Horro Mediano / Horno Mediano / Hervo Mediano	Huevo Mediano
	Pilatano Maduro / Píñano Maduro	Plátano Maduro
	Fitjol Rojo / Frijol Rojo a Granad	Frijol Rojo a Granel
	Acerle Vegetal	Aceite Vegetal
	Leche Pasteurizada India en Bolos	Leche pasteurizada fluida en bolsa
	Leche de Menudo	Manteca Comestible de Origen Vegetal
	Tomate Pecа / Teca	Tomate Pera / Yuca
	Anicar Blanca	Azúcar Blanca
	Pan Molido Blanco	Pan Molde Blanco
	Jugo de Naranja (Bolsa 500 ml)	Jugo de Naranja (500ml)

5.2 Mercados
	Texto ilegible	Corrección
	Mercado Zonal Belén, Comayaguela	mercado zonal belén, comayagüela
	Ahorro Ferias del Pueblo / Ahorro Feria del Pueblo	ahorro ferias del pueblo
	Feria del Agricultor y El Artesano	feria del agricultor y el artesano
	Supermercados Tegucigalpa / Supermercados Tequiegala	supermercados tegucigalpa
	Supermercado San Pedro Sula / Supermercados San Pedro Sula	supermercados san pedro sula
	Mercado El Rápido San Pedro Sula / Mercado El Rapido San Pedro Sula	mercado el rápido san pedro sula
	Feria Agropecuaria de Villanueva / Feria Agropecuaria Villanueva	feria agropecuaria de villanueva
	Mercados San Isidro, las Americas y Colón	mercado san isidro, las americas y colón
	
6. CHECKLIST FINAL ANTES DE ENTREGAR
	-¿Se procesaron todas las páginas del PDF?
	-¿Se incluyó al menos una entrada JSON por cada tabla encontrada?
	-¿Todos los valores están entre comillas dobles (strings)?
	-¿Los campos sin dato tienen "null" y su "error" correspondiente?
	-¿Los nombres de mercado están normalizados según la lista oficial?
	-¿Los nombres de producto y unidades siguen el catálogo estándar?
	-¿Las fechas están en formato DD-MM-AAAA?
	-¿El campo "error" usa el formato "pag X - nombre_del_archivo.pdf"?
	-¿No se agrupó ni resumió ninguna fila?
	-¿El JSON es válido y parseable?

"""

In [0]:
apiKey = dbutils.secrets.get(scope = "deepApiKey", key = "ApiKey")


def api_execute(file_name,prompt):
    #1 Read File Properties
    reader = PdfReader(file_name)
    
    file_content = ""
    for page in reader.pages:
        text = page.extract_text()
        if text:
            file_content += text + "\n"

  
    # 2. Inicializa el cliente e ingresa tu API key directamente aquí
    client = OpenAI(
        api_key=apiKey,  # <-- AQUÍ COLOCAS TU API KEY
        base_url="https://api.deepseek.com"
    )

    
    # 3. Envía la solicitud con tu prompt y el contenido del archivo       

    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=[
            {
                "role": "user",
                "content": f"{prompt}\n\n{file_content}",
            },
        ],
        stream=False
        #reasoning_effort="high",
        #extra_body={"thinking": {"type": "enabled"}},
    )

    query_response=response.choices[0].message.content

    
    return query_response

In [0]:
#Execute the PDF reading for OCR in Deepseek and save the results to a dataframe

def readFileContainer(prompt):
    # List files in a workspace directory
    for file in wkc.workspace.list(
        "/Workspace/Users/jarodriguezv91@gmail.com/Product_Management_D_E_Portfolio/Products_Management/src/PDF_Files"
    ):
        file_name = file.path
        
        query = f"SELECT 1 FROM products.landing.docs_name WHERE docs_name = '{file_name}'"

        df = spark.sql(query)

        if df.isEmpty():
            query = f"INSERT INTO products.landing.docs_name(docs_name) VALUES ('{file_name}')"
            df = spark.sql(query)
            json_tbl=api_execute(file_name, prompt)
            print("The file was read and processed")
            return json_tbl
        else:
            print("The file was already processed")



In [0]:
query_json=readFileContainer(prompt)

In [0]:
%sql
--delete from products.landing.docs_name	

select * from products.landing.docs_name	

In [0]:
print(query_json)

In [0]:
# Formating the JSON into String FORMAT

data = query_json
data_all_string= [{k: str(v) for k, v in d.items()} for d in data]

print(json.dumps(data_all_string,indent=5))

In [0]:
# Defining the column order for dataset

column_order = [
    "nombre_mercado",
    "nombre_producto",
    "unidad_medida",
    "inicio_fecha_precios",
    "fin_fecha_precios",
    "precio_inicio",
    "precio_fin",
    "error",
]


df_products = spark.createDataFrame(data_all_string)
df_product_list = df_products.select(column_order)

In [0]:
#Adding timestamp to the dataset

df_product_list_bronze = df_product_list.withColumn("ingesta_timestamp", current_timestamp())



In [0]:
#Sending the dataset to be written in the table

df_product_list_bronze.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("products.landing.product_list_markets_landing")

In [0]:
%sql
--Checking Bronze table data 

select * from products.landing.product_list_markets_landing

In [0]:
%sql
delete from products.landing.product_list_markets_landing